In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, matthews_corrcoef

In [2]:
df = pd.read_csv("morganFP_2048.csv")
df

,0,1,2,3,4,5,6,7,8,9,...,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
756,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
757,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
758,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
df_class = pd.read_csv("ido_tdo.csv" , index_col=0)
df_class

,ido_tdo
0,AA
1,II
2,AI
3,II
4,II
...,...
755,II
756,II
757,IA
758,AI


In [4]:
y = df_class.iloc[:,-1]
y

0      AA
1      II
2      AI
3      II
4      II
       ..
755    II
756    II
757    IA
758    AI
759    AA
Name: ido_tdo, Length: 760, dtype: object

In [5]:
X = df.iloc[:, :]
X

,0,1,2,3,4,5,6,7,8,9,...,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
756,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
757,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
758,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
#Random-splitting
X_train, X_test, y_train, y_test = train_test_split(X,y, stratify=y, random_state=42 , test_size=0.2)

In [8]:
model = SVC()
params = {
    "C": [0.1, 1, 10, 100, 1000],
    "gamma": ["scale", "auto", 1, 0.1, 0.01, 0.001],
    "kernel": ["linear", "rbf", "poly", "sigmoid"]
}
grid = GridSearchCV(model, params, cv=5, scoring='matthews_corrcoef', n_jobs=-1)
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100, 1000],
                         'gamma': ['scale', 'auto', 1, 0.1, 0.01, 0.001],
                         'kernel': ['linear', 'rbf', 'poly', 'sigmoid']},
             scoring='matthews_corrcoef')

In [9]:
#For random-splitting
grid.best_estimator_

SVC(C=10, gamma=0.01, kernel='sigmoid')

In [10]:
clf = SVC(probability=True, C=10, gamma= 0.01, kernel='sigmoid', random_state=42)

In [11]:
clf.fit(X_train, y_train)

SVC(C=10, gamma=0.01, kernel='sigmoid', probability=True, random_state=42)

In [12]:
ypred = clf.predict(X_test)

In [13]:
print("svm morgan 2048")
print(classification_report(y_test , ypred))
print(confusion_matrix(y_test , ypred))
print(roc_auc_score(y_test, clf.predict_proba(X_test) , multi_class='ovr'))
print(matthews_corrcoef(y_test, ypred))

svm morgan 2048
              precision    recall  f1-score   support

          AA       0.62      0.71      0.67        42
          AI       0.70      0.62      0.66        34
          IA       0.54      0.44      0.48        16
          II       0.80      0.82      0.81        60

    accuracy                           0.70       152
   macro avg       0.67      0.65      0.65       152
weighted avg       0.70      0.70      0.70       152

[[30  8  3  1]
 [ 5 21  0  8]
 [ 6  0  7  3]
 [ 7  1  3 49]]
0.8894158174298391
0.5789135129159534


# **This gives the best score for external test set**

In [14]:
#External test:
df_test = pd.read_csv("test_morganFP_2048.csv")
df_test_class = pd.read_csv("testdata_target.csv")
y_t = df_test_class.iloc[:,-1]
X_t = df_test.iloc[:, :]
yt_pred = clf.predict(X_t)
print("svm")
print(classification_report(y_t , yt_pred))
print(confusion_matrix(y_t , yt_pred))
print(roc_auc_score(y_t, clf.predict_proba(X_t) , multi_class='ovr'))
print(matthews_corrcoef(y_t, yt_pred))

svm
              precision    recall  f1-score   support

          AA       0.00      0.00      0.00         2
          AI       0.64      0.52      0.57        27
          IA       0.00      0.00      0.00         8
          II       0.74      0.89      0.81        74

    accuracy                           0.72       111
   macro avg       0.34      0.35      0.35       111
weighted avg       0.65      0.72      0.68       111

[[ 0  0  0  2]
 [ 0 14  0 13]
 [ 0  0  0  8]
 [ 0  8  0 66]]
0.7250450151709443
0.3493185820172939


/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

# **Using class_weight='balanced'**

In [15]:
model = SVC(class_weight='balanced')
params = {
    "C": [0.1, 1, 10, 100, 1000],
    "gamma": ["scale", "auto", 1, 0.1, 0.01, 0.001],
    "kernel": ["linear", "rbf", "poly", "sigmoid"]
}
grid = GridSearchCV(model, params, cv=5, scoring='matthews_corrcoef', n_jobs=-1)
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SVC(class_weight='balanced'), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100, 1000],
                         'gamma': ['scale', 'auto', 1, 0.1, 0.01, 0.001],
                         'kernel': ['linear', 'rbf', 'poly', 'sigmoid']},
             scoring='matthews_corrcoef')

In [16]:
grid.best_estimator_

SVC(C=100, class_weight='balanced', gamma='auto')

In [17]:
clf = SVC(probability=True, C=100, class_weight='balanced', gamma='auto', random_state=42)

In [18]:
clf.fit(X_train, y_train)

SVC(C=100, class_weight='balanced', gamma='auto', probability=True,
    random_state=42)

In [19]:
ypred = clf.predict(X_test)

In [20]:
print("svm morgan 2048, balanced")
print(classification_report(y_test , ypred))
print(confusion_matrix(y_test , ypred))
print(roc_auc_score(y_test, clf.predict_proba(X_test) , multi_class='ovr'))
print(matthews_corrcoef(y_test, ypred))

svm morgan 2048, balanced
              precision    recall  f1-score   support

          AA       0.62      0.67      0.64        42
          AI       0.69      0.71      0.70        34
          IA       0.47      0.50      0.48        16
          II       0.85      0.78      0.82        60

    accuracy                           0.70       152
   macro avg       0.66      0.66      0.66       152
weighted avg       0.71      0.70      0.71       152

[[28 10  3  1]
 [ 5 24  0  5]
 [ 6  0  8  2]
 [ 6  1  6 47]]
0.894260333764376
0.584897161402866


In [21]:
#External test:
df_test = pd.read_csv("test_morganFP_2048.csv")
df_test_class = pd.read_csv("testdata_target.csv")
y_t = df_test_class.iloc[:,-1]
X_t = df_test.iloc[:, :]
yt_pred = clf.predict(X_t)
print("svm")
print(classification_report(y_t , yt_pred))
print(confusion_matrix(y_t , yt_pred))
print(roc_auc_score(y_t, clf.predict_proba(X_t) , multi_class='ovr'))
print(matthews_corrcoef(y_t, yt_pred))

svm
              precision    recall  f1-score   support

          AA       0.00      0.00      0.00         2
          AI       0.61      0.41      0.49        27
          IA       0.00      0.00      0.00         8
          II       0.72      0.91      0.80        74

    accuracy                           0.70       111
   macro avg       0.33      0.33      0.32       111
weighted avg       0.63      0.70      0.65       111

[[ 0  0  0  2]
 [ 0 11  0 16]
 [ 0  0  0  8]
 [ 0  7  0 67]]
0.7262187577799296
0.2866758404403435


/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

# **SMOTE - Synthetic Minority Over-sampling Technique**

In [22]:
from imblearn.over_sampling import SMOTE

In [23]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [24]:
model = SVC()
params = {
    "C": [0.1, 1, 10, 100, 1000],
    "gamma": ["scale", "auto", 1, 0.1, 0.01, 0.001],
    "kernel": ["linear", "rbf", "poly", "sigmoid"]
}
grid = GridSearchCV(model, params, cv=5, scoring='matthews_corrcoef', n_jobs=-1)
grid.fit(X_train_smote, y_train_smote)

GridSearchCV(cv=5, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100, 1000],
                         'gamma': ['scale', 'auto', 1, 0.1, 0.01, 0.001],
                         'kernel': ['linear', 'rbf', 'poly', 'sigmoid']},
             scoring='matthews_corrcoef')

In [25]:
grid.best_estimator_

SVC(C=10)

In [26]:
clf = SVC(probability=True, C=10, random_state=42)

In [27]:
clf.fit(X_train_smote, y_train_smote)

SVC(C=10, probability=True, random_state=42)

In [28]:
ypred = clf.predict(X_test)

In [29]:
print("svm morgan 2048")
print(classification_report(y_test , ypred))
print(confusion_matrix(y_test , ypred))
print(roc_auc_score(y_test, clf.predict_proba(X_test) , multi_class='ovr'))
print(matthews_corrcoef(y_test, ypred))

svm morgan 2048
              precision    recall  f1-score   support

          AA       0.66      0.74      0.70        42
          AI       0.74      0.68      0.71        34
          IA       0.50      0.44      0.47        16
          II       0.82      0.82      0.82        60

    accuracy                           0.72       152
   macro avg       0.68      0.67      0.67       152
weighted avg       0.72      0.72      0.72       152

[[31  8  2  1]
 [ 4 23  0  7]
 [ 6  0  7  3]
 [ 6  0  5 49]]
0.8737152629902031
0.6079079030113342


In [30]:
#External test:
df_test = pd.read_csv("test_morganFP_2048.csv")
df_test_class = pd.read_csv("testdata_target.csv")
y_t = df_test_class.iloc[:,-1]
X_t = df_test.iloc[:, :]
yt_pred = clf.predict(X_t)
print("svm")
print(classification_report(y_t , yt_pred))
print(confusion_matrix(y_t , yt_pred))
print(roc_auc_score(y_t, clf.predict_proba(X_t) , multi_class='ovr'))
print(matthews_corrcoef(y_t, yt_pred))

svm
              precision    recall  f1-score   support

          AA       0.00      0.00      0.00         2
          AI       0.50      0.04      0.07        27
          IA       0.00      0.00      0.00         8
          II       0.67      0.99      0.80        74

    accuracy                           0.67       111
   macro avg       0.29      0.26      0.22       111
weighted avg       0.57      0.67      0.55       111

[[ 0  0  0  2]
 [ 0  1  0 26]
 [ 0  0  0  8]
 [ 0  1  0 73]]
0.6658409152435227
0.05788669009245317


/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

# **Oversample using Adaptive Synthetic (ADASYN) algorithm**

In [31]:
from imblearn.over_sampling import ADASYN

In [32]:
ada = ADASYN(random_state=42)
X_train_ada, y_train_ada = smote.fit_resample(X_train, y_train)

In [33]:
model = SVC()
params = {
    "C": [0.1, 1, 10, 100, 1000],
    "gamma": ["scale", "auto", 1, 0.1, 0.01, 0.001],
    "kernel": ["linear", "rbf", "poly", "sigmoid"]
}
grid = GridSearchCV(model, params, cv=5, scoring='matthews_corrcoef', n_jobs=-1)
grid.fit(X_train_ada, y_train_ada)

GridSearchCV(cv=5, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100, 1000],
                         'gamma': ['scale', 'auto', 1, 0.1, 0.01, 0.001],
                         'kernel': ['linear', 'rbf', 'poly', 'sigmoid']},
             scoring='matthews_corrcoef')

In [34]:
grid.best_estimator_

SVC(C=10)

In [35]:
clf = SVC(probability=True, C=10, random_state=42)

In [36]:
clf.fit(X_train_ada, y_train_ada)

SVC(C=10, probability=True, random_state=42)

In [37]:
ypred = clf.predict(X_test)

In [38]:
print("svm morgan 2048")
print(classification_report(y_test , ypred))
print(confusion_matrix(y_test , ypred))
print(roc_auc_score(y_test, clf.predict_proba(X_test) , multi_class='ovr'))
print(matthews_corrcoef(y_test, ypred))

svm morgan 2048
              precision    recall  f1-score   support

          AA       0.66      0.74      0.70        42
          AI       0.74      0.68      0.71        34
          IA       0.50      0.44      0.47        16
          II       0.82      0.82      0.82        60

    accuracy                           0.72       152
   macro avg       0.68      0.67      0.67       152
weighted avg       0.72      0.72      0.72       152

[[31  8  2  1]
 [ 4 23  0  7]
 [ 6  0  7  3]
 [ 6  0  5 49]]
0.8737152629902031
0.6079079030113342


In [39]:
#External test:
df_test = pd.read_csv("test_morganFP_2048.csv")
df_test_class = pd.read_csv("testdata_target.csv")
y_t = df_test_class.iloc[:,-1]
X_t = df_test.iloc[:, :]
yt_pred = clf.predict(X_t)
print("svm")
print(classification_report(y_t , yt_pred))
print(confusion_matrix(y_t , yt_pred))
print(roc_auc_score(y_t, clf.predict_proba(X_t) , multi_class='ovr'))
print(matthews_corrcoef(y_t, yt_pred))

svm
              precision    recall  f1-score   support

          AA       0.00      0.00      0.00         2
          AI       0.50      0.04      0.07        27
          IA       0.00      0.00      0.00         8
          II       0.67      0.99      0.80        74

    accuracy                           0.67       111
   macro avg       0.29      0.26      0.22       111
weighted avg       0.57      0.67      0.55       111

[[ 0  0  0  2]
 [ 0  1  0 26]
 [ 0  0  0  8]
 [ 0  1  0 73]]
0.6658409152435227
0.05788669009245317


/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/linu/anaconda3/envs/embedding/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

# **For Bemis-Murcko scaffold-splitting, run the following cells:**

In [40]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

In [41]:
df_total = pd.read_csv('final_total.csv')
smiles_list = [df_total.loc[i, 'smiles'] for i in range(len(df_total))]

In [42]:
def generate_scaffold(smiles, include_chirality=False):
    """
    Generate the Bemis-Murcko scaffold for a given SMILES string.
    
    Parameters:
    -----------
    smiles : str
        SMILES representation of the molecule
    include_chirality : bool
        Whether to include chirality in the scaffold
    
    Returns:
    --------
    str : Scaffold SMILES string
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return smiles
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(
            mol=mol, 
            includeChirality=include_chirality
        )
        return scaffold
    except:
        return smiles

def scaffold_split(X, y, smiles_list, frac_train=0.8, frac_test=0.2, seed=None):
    """
    Split data by Bemis-Murcko scaffolds.
    
    Parameters:
    -----------
    X : array-like
        Feature matrix (fingerprints)
    y : array-like
        Target labels
    smiles_list : list
        List of SMILES strings for each molecule
    frac_train : float
        Fraction of data for training
    frac_test : float
        Fraction of data for test
    seed : int or None
        Random seed for reproducibility
    
    Returns:
    --------
    tuple : (X_train, X_test, y_train, y_test, 
             idx_train, idx_test)
    """
    n_samples = len(smiles_list)
    
    # Group molecules by scaffold
    scaffolds = defaultdict(list)
    for idx, smiles in enumerate(smiles_list):
        scaffold = generate_scaffold(smiles, include_chirality=True)
        scaffolds[scaffold].append(idx)
    
    # Sort scaffolds by size (largest first)
    sorted_scaffolds = sorted(
        scaffolds.items(), 
        key=lambda x: len(x[1]), 
        reverse=True
    )
    
    # Calculate target sizes
    train_cutoff = int(frac_train * n_samples)
    
    train_idx = []
    test_idx = []
    
    # Assign entire scaffold groups to splits
    for scaffold, indices in sorted_scaffolds:
        if len(train_idx) + len(indices) <= train_cutoff:
            train_idx.extend(indices)
        else:
            test_idx.extend(indices)
    
    # Convert to numpy arrays if needed
    X = np.array(X)
    y = np.array(y)
    
    # Create splits
    X_train = X[train_idx]
    X_test = X[test_idx]
    
    y_train = y[train_idx]
    y_test = y[test_idx]
    
    print(f"Scaffold split results:")
    print(f"  Training:   {len(train_idx)} molecules ({len(train_idx)/n_samples*100:.1f}%)")
    print(f"  Test:       {len(test_idx)} molecules ({len(test_idx)/n_samples*100:.1f}%)")
    print(f"  Unique scaffolds: {len(scaffolds)}")
    
    return (X_train, X_test, 
            y_train, y_test,
            train_idx, test_idx)

In [43]:
# Perform scaffold split
X_train, X_test, y_train, y_test, idx_train, idx_test = scaffold_split(
    X, 
    y, 
    smiles_list,
    frac_train=0.8,
    frac_test=0.2,
    seed=42
)

print(f"\nShape of splits:")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_test: {X_test.shape}, y_test: {y_test.shape}")

Scaffold split results:
  Training:   608 molecules (80.0%)
  Test:       152 molecules (20.0%)
  Unique scaffolds: 250

Shape of splits:
  X_train: (608, 2048), y_train: (608,)
  X_test: (152, 2048), y_test: (152,)


In [44]:
model = SVC()
params = {
    "C": [0.1, 1, 10, 100, 1000],
    "gamma": ["scale", "auto", 1, 0.1, 0.01, 0.001],
    "kernel": ["linear", "rbf", "poly", "sigmoid"]
}
grid = GridSearchCV(model, params, cv=5, scoring='matthews_corrcoef', n_jobs=-1)
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100, 1000],
                         'gamma': ['scale', 'auto', 1, 0.1, 0.01, 0.001],
                         'kernel': ['linear', 'rbf', 'poly', 'sigmoid']},
             scoring='matthews_corrcoef')

In [45]:
#For scaffold-splitting
grid.best_estimator_

SVC(C=1, kernel='sigmoid')

In [46]:
clf = SVC(probability=True, C=1, kernel='sigmoid', random_state=42)
clf.fit(X_train, y_train)
ypred = clf.predict(X_test)

In [47]:
print("svm morgan 2048")
print(classification_report(y_test , ypred))
print(confusion_matrix(y_test , ypred))
print(roc_auc_score(y_test, clf.predict_proba(X_test) , multi_class='ovr'))
print(matthews_corrcoef(y_test, ypred))

svm morgan 2048
              precision    recall  f1-score   support

          AA       0.46      0.74      0.57        35
          AI       0.85      0.72      0.78        39
          IA       0.00      0.00      0.00        11
          II       0.73      0.69      0.71        67

    accuracy                           0.66       152
   macro avg       0.51      0.54      0.51       152
weighted avg       0.65      0.66      0.64       152

[[26  0  0  9]
 [ 9 28  0  2]
 [ 5  0  0  6]
 [16  5  0 46]]
0.8396846701149696
0.5045919533049413


In [48]:
#External test:
df_test = pd.read_csv("test_morganFP_2048.csv")
df_test_class = pd.read_csv("testdata_target.csv")
y_t = df_test_class.iloc[:,-1]
X_t = df_test.iloc[:, :]
yt_pred = clf.predict(X_t)
print("svm")
print(classification_report(y_t , yt_pred))
print(confusion_matrix(y_t , yt_pred))
print(roc_auc_score(y_t, clf.predict_proba(X_t) , multi_class='ovr'))
print(matthews_corrcoef(y_t, yt_pred))

svm
              precision    recall  f1-score   support

          AA       0.00      0.00      0.00         2
          AI       1.00      0.04      0.07        27
          IA       0.00      0.00      0.00         8
          II       0.67      1.00      0.80        74

    accuracy                           0.68       111
   macro avg       0.42      0.26      0.22       111
weighted avg       0.69      0.68      0.55       111

[[ 0  0  0  2]
 [ 0  1  0 26]
 [ 0  0  0  8]
 [ 0  0  0 74]]
0.7064029087904276
0.1369745414740455
